In [1]:
# Imports
import sys, time, math, json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, str(Path.cwd().parent))

from common.utils import set_seed, get_device, count_parameters
from common.datasets import (FASHION_MNIST_CLASSES, FASHION_MNIST_SHAPE, 
                             load_fashion_mnist, load_or_create_split,
                             pixel_stats, to_tensor_dataset,
                             make_train_loader, make_eval_loaders)
from common.training import fit
from common.evaluation import collect_outputs, evaluate_model, format_results_table
from common.plotting import setup_style, save_fig, plot_learning_curves, plot_confusion_matrices
from E1.config import CONFIG, SPLIT_PATH, NUM_CLASSES

set_seed(CONFIG["seed"])
DEVICE = get_device()

E2_CONFIG = dict(
  embed_dim=64, 
  depth=2, 
  num_heads=4, 
  mlp_ratio=2, 
  dropout=0.1, 
  pool="cls", 
  head="softmax"
)

RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)
FIG_DIR = Path("figures"); FIG_DIR.mkdir(exist_ok=True)
setup_style()

In [2]:
# load data
train_images, train_labels, test_images, test_labels = load_fashion_mnist(CONFIG["data_dir"])
train_idx, val_idx = load_or_create_split(train_labels, SPLIT_PATH, CONFIG["seed"], CONFIG["val_ratio"])
X_train, y_train = train_images[train_idx], train_labels[train_idx]
X_val, y_val = train_images[val_idx], train_labels[val_idx]
X_test, y_test = test_images, test_labels
mean, std = pixel_stats(X_train)
datasets = {
    "train": to_tensor_dataset(X_train, y_train, mean, std),
    "val": to_tensor_dataset(X_val, y_val, mean, std),
    "test": to_tensor_dataset(X_test, y_test, mean, std),
}

train_loader = make_train_loader(datasets["train"], CONFIG["batch_size"], CONFIG["seed"])
eval_loaders = make_eval_loaders(datasets, CONFIG["batch_size"])

## MSA

Input $X \in \mathbb{R}^{B \times N \times D}$, gồm $N$ token, mỗi token có $D$ chiều, chia thành $h$ head với $d = D/h$, $B$ là số lượng sample.

Giả sử ta đem các giá trị vào một Tensor, với B là số lượng sample, N là số lượng token, thì:

$$Q = XW_Q,\quad K = XW_K,\quad V = XW_V$$
$$\text{head}_i = \text{softmax}\!\left(\frac{Q_i K_i^\top}{\sqrt{d}}\right) V_i$$
$$\text{MSA}(X) = \text{Concat}(\text{head}_1,\dots,\text{head}_h)\, W_O$$

Cụ thể, $Q,K,V \in \mathbb{R}^{B \times N \times d} \Rightarrow QK^\top \in \mathbb{R}^{B \times N \times N}$ hoặc $Q,K,V \in \mathbb{R}^{B \times h \times N \times d} \Rightarrow QK^\top \in \mathbb{R}^{B \times h \times N \times N}$ 

$\text{head}_i \in \mathbb{R}^{B \times N \times d}$

In [3]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, attn_dropout: float = 0.0, bias: bool = True):
        super().__init__()
        self.embed_dim = embed_dim                # D
        self.num_heads = num_heads                # h
        self.head_dim = embed_dim // num_heads    # D/h
        self.scale = 1 / math.sqrt(self.head_dim)
        self.qkv = nn.Linear(embed_dim, embed_dim * 3, bias=bias) # W_q, W_k, W_v combined
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=bias) # Output projection
        self.attn_dropout = nn.Dropout(attn_dropout)

    def forward(self, x, return_attn: bool = False):
        B, N, D = x.shape

        # 1. Tính Q, K, V
        qkv = self.qkv(x)  # (B, N, 3*D)
        qkv = qkv.reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)  # (3, B, h, N, D/h)
        q, k, v = qkv[0], qkv[1], qkv[2]  # Each is (B, h, N, D/h)

        # 2. Tính attention scores bằng einsum
        attn_scores = torch.einsum('bhqd,bhkd->bhqk', q, k) * self.scale  # (B, h, N, N)
        attn_probs = F.softmax(attn_scores, dim=-1)
        attn_probs = self.attn_dropout(attn_probs)

        # 3. Tính weighted sum của V
        attn_output = torch.einsum('bhqk,bhkd->bhqd', attn_probs, v)  # (B, h, N, D/h)
        attn_output = attn_output.permute(0, 2, 1, 3).reshape(B, N, D)  # (B, N, D)

        # 4. Áp dụng output projection
        output = self.out_proj(attn_output)

        if return_attn:
            return output, attn_probs
        return output


In [4]:
class MSAPyTorch(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, attn_dropout: float = 0.0, bias: bool = True):
        super().__init__()
        self.mha = nn.MultiheadAttention(
            embed_dim=embed_dim, 
            num_heads=num_heads, 
            dropout=attn_dropout, 
            bias=bias, 
            batch_first=True # Để đầu vào có dạng (B, N, D) thay vì (N, B, D)
        )

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        attn_output, attn_weights = self.mha(x, x, x, need_weights=return_attn, average_attn_weights=False)
        if return_attn:
            return attn_output, attn_weights
        return attn_output

In [5]:
ATTENTIONS = {
  "scratch": MultiHeadSelfAttention,
  "torch": MSAPyTorch,
}

# Ánh xạ hậu tố key: scratch -> torch
_MSA_KEY_MAP = {
  "qkv.weight": "mha.in_proj_weight",
  "qkv.bias": "mha.in_proj_bias",
  "out_proj.weight": "mha.out_proj.weight",
  "out_proj.bias": "mha.out_proj.bias",
}

def scratch_to_torch_state_dict(state_dict: dict) -> dict:
    torch_state_dict = {}
    for key, value in state_dict.items():
        # Nếu key kết thúc bằng một hậu tố trong _MSA_KEY_MAP, ánh xạ sang key tương ứng của torch
        for suffix, torch_key in _MSA_KEY_MAP.items():
            if key.endswith(suffix):
                torch_state_dict[key[:-len(suffix)] + torch_key] = value
                break
        else:
            torch_state_dict[key] = value
    return torch_state_dict

In [6]:
# Kiểm tra hai mô hình Attention

@torch.no_grad()
def check_scratch_and_torch(B=4, N=20, D=64, h=4, atol=1e-5):
    set_seed(0)
    scratch = MultiHeadSelfAttention(embed_dim=D, num_heads=h).eval()
    torch_msa = MSAPyTorch(embed_dim=D, num_heads=h).eval()
    torch_msa.load_state_dict(scratch_to_torch_state_dict(scratch.state_dict()))

    # Tạo đầu vào ngẫu nhiên
    x = torch.randn(B, N, D)

    # Kiểm tra đầu ra của scratch và torch có gần nhau không
    out_scratch, attn_scratch = scratch(x, return_attn=True)
    out_torch, attn_torch = torch_msa(x, return_attn=True)
    assert torch.allclose(out_scratch, out_torch, atol=atol) and torch.allclose(attn_scratch, attn_torch, atol=atol), "Outputs are not close!"

    # Tính số tham số
    n_s, n_t = count_parameters(scratch), count_parameters(torch_msa)
    print(f"Number of parameters - Scratch: {n_s}, Torch: {n_t}")

    # Kiểm tra số tham số phải bằng nhau và bằng 4D^2 + 4D
    assert n_s == n_t == 4 * D**2 + 4 * D, "Number of parameters is not correct!"

check_scratch_and_torch()

Number of parameters - Scratch: 16640, Torch: 16640


In [7]:
# Đo tốc độ chạy của hai mô hình Attention
def sync(device):
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elif device.type == 'mps':
        torch.mps.synchronize()

def bench_attention(cls, N, D=64, h=4, B=128, iters=50, warmup=10, device=DEVICE):
    set_seed(0)
    module = cls(embed_dim=D, num_heads=h).to(device).train()
    x = torch.randn(B, N, D, device=device, requires_grad=True)
    for i in range(warmup + iters):
        if (i == warmup):
            sync(device)
            t0 = time.perf_counter()
        module(x).sum().backward()
    sync(device)
    t1 = time.perf_counter()
    return (t1 - t0) / iters * 1e3  # Convert to milliseconds

# N = số token với ảnh 28×28:
#   16  = patch size 7 → lưới 4×4
#   28  = mỗi hàng là 1 token
#   49  = patch size 4 → lưới 7×7
#   196 = patch size 2 → lưới 14×14
for N in (16, 28, 49, 196):
    times = {
        name: bench_attention(cls, N) for name, cls in ATTENTIONS.items()
    }
    print(f"N={N:>4}: " + "  ".join(f"{k}={v:6.2f} ms" for k, v in times.items()))

N=  16: scratch=  0.74 ms  torch=  0.84 ms
N=  28: scratch=  1.33 ms  torch=  1.19 ms
N=  49: scratch=  2.65 ms  torch=  2.82 ms
N= 196: scratch= 26.56 ms  torch= 27.58 ms


In [8]:
class FeedForward(nn.Module):
    """Áp dụng độc lập cho từng token: [B, N, D] -> [B, N, D]"""

    def __init__(self, embed_dim, mlp_ratio=2, dropout=0.1):
        super().__init__()
        hidden_dim = embed_dim * mlp_ratio
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(p=dropout)
        )

    def forward(self, x):
        return self.net(x)

In [9]:
class EncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=2, dropout=0.1, attention="scratch"):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attention = ATTENTIONS[attention](embed_dim=embed_dim, num_heads=num_heads, attn_dropout=dropout)
        self.drop = nn.Dropout(p=dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = FeedForward(embed_dim=embed_dim, mlp_ratio=mlp_ratio, dropout=dropout)

    def forward(self, x, return_attn=False):
        # Pre-norm and attention
        if return_attn:
            attn_out, attn_weights = self.attention(self.norm1(x), return_attn=True)
        else:
            attn_out, attn_weights = self.attention(self.norm1(x)), None
        
        x = x + self.drop(attn_out)
        x = x + self.ffn(self.norm2(x))

        if return_attn:
            return x, attn_weights
        return x

In [10]:
from E1.classifiers import SoftmaxClassifier, MLPClassifier

class TransformerClassifier(nn.Module):
    def __init__(self, tokenizer, attention="scratch", embed_dim=64, depth=2, num_heads=4, 
                 mlp_ratio=2, dropout=0.1, pool="cls", head="softmax", use_pos=True):
        super().__init__()
        self.tokenizer = tokenizer
        self.pool = pool
        num_tokens = tokenizer.num_tokens
        if pool == "cls":
            num_tokens += 1

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim)) if pool == "cls" else None
        self.pos_embed = nn.Parameter(torch.zeros(1, num_tokens, embed_dim)) if use_pos else None
        self.pos_drop = nn.Dropout(p=dropout) if use_pos else None
        self.blocks = nn.ModuleList([
            EncoderBlock(embed_dim=embed_dim, num_heads=num_heads, 
                         mlp_ratio=mlp_ratio, dropout=dropout, attention=attention)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        if head == "softmax":
            self.head = SoftmaxClassifier(input_dim=embed_dim, num_classes=NUM_CLASSES)
        else:
            self.head = MLPClassifier(input_dim=embed_dim, num_classes=NUM_CLASSES, 
                                      hidden_dims=(embed_dim * mlp_ratio,), dropout=dropout)
        
        if self.pos_embed is not None:
            nn.init.trunc_normal_(self.pos_embed, std=0.02)
        if self.cls_token is not None:
            nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x, return_attn: bool = False):
        tokens = self.tokenizer(x)
        
        if self.cls_token is not None:
            cls_tokens = self.cls_token.expand(tokens.size(0), -1, -1)
            tokens = torch.cat((cls_tokens, tokens), dim=1)
        
        if self.pos_embed is not None:
            tokens = tokens + self.pos_embed
            tokens = self.pos_drop(tokens)

        attn_maps = []
        for block in self.blocks:
            if return_attn:
                tokens, attn = block(tokens, return_attn=True)
                attn_maps.append(attn)
            else:
                tokens = block(tokens)
        
        tokens = self.norm(tokens)
        
        if self.pool == "cls":
            tokens = tokens[:, 0]
        else:
            tokens = tokens.mean(dim=1)

        logits = self.head(tokens)
        if return_attn:
            return logits, attn_maps
        return logits

    @torch.no_grad()
    def predict_proba(self, x):
        self.eval()
        return torch.softmax(self(x), dim=-1)

In [11]:
# Tokenizer

class PatchEmbedding(nn.Module):
    """Tokenizer theo phương pháp Patching, default là patch_size=7, embed_dim=64"""
    def __init__(self, patch_size=7, embed_dim=64, img_size=28, in_channels=1):
        super().__init__()
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.img_size = img_size
        self.in_channels = in_channels
        self.num_tokens = (img_size // patch_size) ** 2
        self.proj = nn.Linear(patch_size * patch_size * in_channels, embed_dim)

    def forward(self, x):
        B, C, H, W = x.shape
        p = self.patch_size

        # [B, C, H/p, p, W/p, p] -> [B, H/p, W/p, C, p, p]
        x = x.reshape(B, C, H // p, p, W // p, p)
        x = x.permute(0, 2, 4, 1, 3, 5)
        x = x.reshape(B, -1, C * p * p)
        return self.proj(x)

class RowEmbedding(nn.Module):
    """Tokenizer theo phương pháp Row Embedding, với token size đúng bằng 28pixel - tức là một dòng của mỗi ảnh"""
    def __init__(self, embed_dim=64, img_size=28, in_channels=1):
        super().__init__()
        self.num_tokens = img_size # mỗi dòng là một token
        self.proj = nn.Linear(img_size * in_channels, embed_dim) # mỗi dòng của ảnh được flatten và ánh xạ vào không gian embedding
        # tương đương L(C·W → D)

    def forward(self, x):
        B, C, H, W = x.shape
        # Đưa H lên trước C: [B, C, H, W] -> [B, H, C, W]
        # [B, H, C, W] -> [B, H, C*W]
        # rồi chiếu sang không gian embedding D
        x = x.permute(0, 2, 1, 3)  # [B, C, H, W] -> [B, H, C, W]
        x = x.reshape(B, H, C * W)  # [B, H, C, W] -> [B, H, C*W]
        return self.proj(x)

from E1.classifiers import conv_block  

class CNNStemEmbedding(nn.Module):
    """Tokenizer theo phương pháp CNN Stem"""
    def __init__(self, embed_dim=64, img_size=28, input_channels=1):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(input_channels, embed_dim // 2),
            conv_block(embed_dim // 2, embed_dim),
        )
        self.num_tokens = (img_size // 4) ** 2  # giả sử CNN stem giảm kích thước ảnh xuống 1/4
        self.proj = nn.Linear(embed_dim, embed_dim)  # chiếu các token CNN stem sang không gian embedding D


    def forward(self, x):
        f = self.features(x)  # [B, C', H', W']
        # flatten feature map: [B, C', H', W'] -> [B, H'*W', C']
        B, C, H, W = f.shape
        f = f.reshape(B, C, H * W)
        f = f.permute(0, 2, 1)  # [B, C, H*W] -> [B, H*W, C]
        return self.proj(f)

TOKENIZERS = {
    "patch7": lambda D: PatchEmbedding(patch_size=7, embed_dim=D),
    "row":    lambda D: RowEmbedding(embed_dim=D),
    "cnn":    lambda D: CNNStemEmbedding(embed_dim=D),
}

In [12]:
# Test 1 shape và bảng tham số

for name, make in TOKENIZERS.items():
    set_seed(0)
    model = TransformerClassifier(tokenizer=make(64))
    x = torch.randn(8, 1, 28, 28)

    print(f"Testing tokenizer: {name}")
    assert model(x).shape == (8, 10), "Output shape is incorrect"

    print(f"{'name':<10} {'parameters':>8}")
    for name, part in [("tokenizer", model.tokenizer), ("1 block", model.blocks[0]),
                      ("  ├ attn", model.blocks[0].attention), ("  └ ffn", model.blocks[0].ffn),
                      ("head", model.head), ("TOTAL", model)]:
        print(f"{name:<10} {count_parameters(part):>8,}")

    print()

Testing tokenizer: patch7
name       parameters
tokenizer     3,200
1 block      33,472
  ├ attn     16,640
  └ ffn      16,576
head            650
TOTAL        72,074

Testing tokenizer: row
name       parameters
tokenizer     1,856
1 block      33,472
  ├ attn     16,640
  └ ffn      16,576
head            650
TOTAL        71,498

Testing tokenizer: cnn
name       parameters
tokenizer    22,976
1 block      33,472
  ├ attn     16,640
  └ ffn      16,576
head            650
TOTAL        93,962



In [13]:
# Test 2 — scratch và torch cho cùng logits khi dùng chung trọng số

@torch.no_grad()
def check_classifier_equivalence(atol=1e-5, tokenizer_name="patch7"):
    set_seed(0)
    tok = TOKENIZERS[tokenizer_name](64)
    m_scratch = TransformerClassifier(tok, attention="scratch").eval()
    m_torch = TransformerClassifier(tok, attention="torch").eval()
    m_torch.load_state_dict(scratch_to_torch_state_dict(m_scratch.state_dict()))   # strict=True
    x = torch.randn(8, 1, 28, 28)
    diff = (m_scratch(x) - m_torch(x)).abs().max().item()
    print(f"{tokenizer_name:<8} max |Δlogits| = {diff:.2e}")
    assert diff < atol, f"Difference {diff:.2e} exceeds tolerance {atol}"

In [14]:
for tokenizer_name in TOKENIZERS.keys():
    check_classifier_equivalence(tokenizer_name=tokenizer_name)

patch7   max |Δlogits| = 7.15e-07
row      max |Δlogits| = 6.41e-07
cnn      max |Δlogits| = 5.51e-07


In [15]:
# Test 3 — overfit một batch

for name, make in TOKENIZERS.items():
    print(f"Testing tokenizer: {name}")
    xb, yb = next(iter(train_loader))
    xb, yb = xb[:64], yb[:64]
    D = 64
    tok = make(D)
    set_seed(0)
    model = TransformerClassifier(tok)
    opt = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])

    for step in range(200):
      model.train()
      opt.zero_grad()
      logits = model(xb)
      loss = F.cross_entropy(logits, yb)
      loss.backward()
      opt.step()
      if step % 50 == 0 or step == 199:
        print(f"Step {step:>3}: loss = {loss.item():.4f}")
    assert loss.item() < 0.05, f"Không overfit được 1 batch: loss = {loss.item():.4f}"
    print()

Testing tokenizer: patch7
Step   0: loss = 2.5879
Step  50: loss = 0.2056
Step 100: loss = 0.0484
Step 150: loss = 0.0276
Step 199: loss = 0.0094

Testing tokenizer: row
Step   0: loss = 2.3514
Step  50: loss = 0.1912
Step 100: loss = 0.0281
Step 150: loss = 0.0261
Step 199: loss = 0.0071

Testing tokenizer: cnn
Step   0: loss = 2.5510
Step  50: loss = 0.4446
Step 100: loss = 0.0658
Step 150: loss = 0.0420
Step 199: loss = 0.0090



## Training

In [16]:
import copy

@torch.no_grad()
def measure_latency(model, iters=200, warmup=10, device=torch.device("cpu")):
    """Đo độ trễ của một model với batch=1. Chọn device mặc định là cpu để gần với kịch bản chạy trên thiết bị."""
    model = copy.deepcopy(model).to(device).eval()
    x = torch.randn(1, 1, 28, 28, device=device)

    # warm up
    for _ in range(warmup):
        _ = model(x)

    sync(device)
    t0 = time.perf_counter()
    
    for _ in range(iters):
        _ = model(x)
    
    sync(device)
    t1 = time.perf_counter()

    return 1000 * (t1 - t0) / iters  # latency in milliseconds

In [ ]:
def run_experiment(tokenizer_name, attention="scratch", init_state=None, seed=CONFIG['seed']):
    assert attention in ["scratch", "torch"], "Attention must be either 'scratch' or 'torch'"
    name = f"{tokenizer_name}_{attention}"
    set_seed(seed)
    D = E2_CONFIG["embed_dim"]
    model = TransformerClassifier(
        tokenizer=TOKENIZERS[tokenizer_name](D),
        attention=attention,
        embed_dim=D,
        depth=E2_CONFIG["depth"],
        num_heads=E2_CONFIG["num_heads"],
        mlp_ratio=E2_CONFIG["mlp_ratio"],
        dropout=E2_CONFIG["dropout"],
        pool=E2_CONFIG["pool"],
        head=E2_CONFIG["head"]
    )

    if init_state is not None:
        model.load_state_dict(init_state)

    init_snapshot = {
        k: v.detach().clone() for k, v in model.state_dict().items()
    }

    train_loader = make_train_loader(datasets["train"], CONFIG["batch_size"], CONFIG["seed"])
    t0 = time.perf_counter()
    history, best_epoch = fit(
        model=model,
        train_loader=train_loader,
        val_loader=eval_loaders["val"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
        seed=seed,
        patience=CONFIG["patience"],
        device=DEVICE,
    )
    history["train_time"] = time.perf_counter() - t0

    row = evaluate_model(
        model=model,
        name=name,
        best_epoch=best_epoch,
        train_time=history["train_time"],
        eval_loaders=eval_loaders,
        device=DEVICE,
    )
    row.update(
        tokenizer=tokenizer_name,
        attention=attention,
        num_tokens=model.tokenizer.num_tokens,
        sec_per_epoch=float(np.mean(history["epoch_time"])),
        latency_ms=measure_latency(model),
    )

    outputs = collect_outputs(
        model=model,
        dataloader=eval_loaders["test"],
        device=DEVICE,
    )

    return row, history, outputs, model, init_snapshot

In [20]:
results, histories, outputs, models = [], {}, {}, {}
results_path, hist_path = RESULTS_DIR / "results.json", RESULTS_DIR / "histories.json"
results = json.loads(results_path.read_text()) if results_path.exists() else []
histories = json.loads(hist_path.read_text()) if hist_path.exists() else {}
done = {r["model"] for r in results}

def save_progress():
    results_path.write_text(json.dumps(results, indent=2))
    hist_path.write_text(json.dumps(histories, indent=2))

def record(name, row, hist, out, model):
    results.append(row)
    done.add(name)
    histories[name] = hist
    outputs[name] = out
    models[name] = model
    save_progress()
    torch.save(model.state_dict(), RESULTS_DIR / f"model_{name}.pt")
    np.savez(RESULTS_DIR / f"outputs_{name}.npz", y_true=out[0], y_pred=out[1], probs=out[2])

In [ ]:
for tokenizer_name in TOKENIZERS:
    scratch_name, torch_name = f"{tokenizer_name}_scratch", f"{tokenizer_name}_torch"
    init_path = RESULTS_DIR / f"init_state_{scratch_name}.pt"

    # (1) Chạy toàn bộ tokenizer với cùng attention 'scratch' trước
    if scratch_name in done:
        print("Đã chạy ", scratch_name)
    else:
        row, hist, out, model, init_st = run_experiment(tokenizer_name, attention="scratch")
        torch.save(init_st, init_path)
        record(scratch_name, row, hist, out, model)

    # (2) Chạy toàn bộ với attention 'torch'
    if torch_name in done:
        print("Đã chạy ", torch_name)
    else:
        init_st = torch.load(init_path)
        row, hist, out, model, _ = run_experiment(tokenizer_name, attention="torch", init_state=scratch_to_torch_state_dict(init_st))
        record(torch_name, row, hist, out, model)
    
    # Dùng format_results_table ở E1
    print(format_results_table(results))

Epoch 1/50 - Train loss: 0.8320, Train acc: 0.7056 - Val loss: 0.5506, Val acc: 0.8016
Epoch 2/50 - Train loss: 0.5462, Train acc: 0.8039 - Val loss: 0.4604, Val acc: 0.8378
Epoch 3/50 - Train loss: 0.4819, Train acc: 0.8247 - Val loss: 0.4371, Val acc: 0.8431
Epoch 4/50 - Train loss: 0.4488, Train acc: 0.8364 - Val loss: 0.3991, Val acc: 0.8571
Epoch 5/50 - Train loss: 0.4235, Train acc: 0.8443 - Val loss: 0.3898, Val acc: 0.8595
Epoch 6/50 - Train loss: 0.4064, Train acc: 0.8518 - Val loss: 0.3912, Val acc: 0.8587
Epoch 7/50 - Train loss: 0.3947, Train acc: 0.8549 - Val loss: 0.3515, Val acc: 0.8716
Epoch 8/50 - Train loss: 0.3821, Train acc: 0.8593 - Val loss: 0.3630, Val acc: 0.8667
Epoch 9/50 - Train loss: 0.3737, Train acc: 0.8636 - Val loss: 0.3632, Val acc: 0.8693
Epoch 10/50 - Train loss: 0.3610, Train acc: 0.8682 - Val loss: 0.3390, Val acc: 0.8788
Epoch 11/50 - Train loss: 0.3548, Train acc: 0.8682 - Val loss: 0.3407, Val acc: 0.8783
Epoch 12/50 - Train loss: 0.3486, Train a

## Evaluation

In [22]:
print(format_results_table(results))

Model              Params  Best ep  Time(s)    Train      Val     Test
----------------------------------------------------------------------
patch7_scratch     72,074       25     76.6   91.60%   89.52%   88.01%
patch7_torch       72,074       25     77.1   91.39%   89.20%   88.04%
row_scratch        71,498       36    139.7   94.12%   90.64%   89.26%
row_torch          71,498       24     90.3   92.81%   90.57%   89.47%
cnn_scratch        93,962       15    138.2   93.97%   91.93%   91.13%
cnn_torch          93,962       21    178.2   94.78%   92.11%   91.04%


Qua đó ta có thể nhận xét thành hai nhóm:

- Nhóm 1: so sánh với cùng tokenizer nhưng khác nhau ở **scratch** và **torch**:
  - Về số parameters, số lượng tham số của mô hình scratch và torch ở các tokenizer đều giống nhau, khoảng 72k cho patch7x7, 71k cho row chia thành 28 tokens, hoặc 93k cho CNN.
  - Nhưng có sự khác biệt đôi chút về best epoch, tức là epoch có val accuracy cao nhất, cũng là checkpoint được giữ lại.
  - Với tokenizer là patch 7x7 thì mô hình MSA scratch và torch đều không có sự khác nhau về số lượng tham số, best epoch, không có sự khác nhau quá lớn hoặc đáng kể ở thời gian training, train acc/val acc/test acc.
  - Với tokenizer là row 28px thì hai mô hình có sự khác nhau nhiều hơn và thấy rõ mô hình viết tay scratch sẽ có các giá trị lớn hơn mô hình native, như thông tin best epoch sẽ cho biết tốc độ hội tụ của mô hình native tốt hơn mô hình scratch khoảng 12 epoch, điều này dẫn tới thời gian training của scratch cũng sẽ lớn hơn torch vì early stopping, nếu nhìn vào test acc cuối cùng để đánh giá, thì mô hình torch nhỉnh hơn scratch 0.21 điểm, không quá lớn.
  - Với tokenizer là CNN, tương tự với cách so sánh ở Row, thì mô hình torch có kết quả kém hơn scratch một chút.

- Nhóm 2: so sánh với cùng loại mô hình nhưng khác tokenizer:
  - Với scratch, patch 7x7 cho test acc thấp nhất khi so với Row và CNN (88.01% < 89.26% < 91.13%), tuy best epoch của Row là 36 (cao nhất) và CNN là 15 (nhất nhất) nhưng cả hai mô hình có hai tokenizer này thì có thời gian training gần sát nhau (139s với 138s), điều này còn phản ánh một điều, tuy dừng sớm nhưng CNN với số lượng tham số nhiều hơn hẳn so với Row đồng thời đi qua hai lớp conv nên thời gian training network này chắc chắn phải chậm hơn các mô hình tokenizer khác.
  - Với torch, xét về test acc sau cùng thì CNN vẫn tốt hơn Row sau đó mới đến Patch, best epoch của ba mô hình có tokenizer này cũng có sai khác không lớn, tuy nhiên, với cùng lập luận trên, ở cùng số epoch thì CNN phải tiêu tốn thời gian training nhiều hơn hẳn các mô hình tokenizer còn lại.

In [ ]:
# Visualize the results

## Nạp lại outputs từ results hoặc từ file
for r in results:
    name = r["model"]
    if name not in outputs:
        d = np.load(RESULTS_DIR / f"outputs_{name}.npz")
        outputs[name] = (d["y_true"], d["y_pred"], d["probs"])

ATTN_COLORS = {"scratch": "#2a78d6", "torch": "#F2A900"}
TOK_COLORS = {"patch7": "#DE7272", "row": "#55A9A0", "cnn": "#F2A900"}

for tok in TOKENIZERS:
    names = [f"{tok}_{a}" for a in ATTENTIONS]
    fig = plot_learning_curves({n: histories[n] for n in names},
                               {n: ATTN_COLORS[n.split("_")[1]] for n in names})
    fig.suptitle(f"{tok}: MSA tự viết vs. nn.MultiheadAttention", y=1.02)
    save_fig(fig, FIG_DIR, f"E2_curves_{tok}_scratch_vs_torch")

fig = plot_learning_curves({t: histories[f"{t}_scratch"] for t in TOKENIZERS}, TOK_COLORS)
fig.suptitle("So sánh tokenization (MSA tự viết)", y=1.02)
save_fig(fig, FIG_DIR, "E2_curves_tokenizers")

PosixPath('figures/E2_curves_tokenizers.png')

![alt text](figures/E2_curves_patch7_scratch_vs_torch.png)
![alt text](figures/E2_curves_row_scratch_vs_torch.png)
![alt text](figures/E2_curves_cnn_scratch_vs_torch.png)
![alt text](figures/E2_curves_tokenizers.png)

Qua ba learning curves của Patch, Row, CNN giữa scratch và torch ở trên, thì ta có thể thấy:
- Đường "val" của các tokenizer ở cả hai phiên bản đều gần như trùng nhau, xác thực kết quả số liệu trên bảng ở trước đó.
- Ở tokenizer Patch 7x7 thì chấm tròn (thể hiện cho early stopping) trùng nhau, còn ở Row thì torch dừng sớm hơn scratch, ngược lại với CNN, scratch dừng sớm hơn torch.
- Trong ba bộ tokenizer, CNN hội tụ nhanh nhất được dựa vào best epoch = 15 so với 25 của patch và 36 của row, đạt khoảng 88% (val cao nhất của patch) ở epoch thứ 3 và sau đó tiếp tục tăng lên đến điểm lớn nhất ở epoch 15. Nhìn qua loss, đường val loss xuống đến đáy ở epoch 15 sau đó tăng nhẹ (có dấu hiệu overfit vì trong khi train loss tiếp tục giảm) và bị cắt đúng lúc nhờ early stopping.
- Trong khi đó, Row thì có val loss gần như đi ngang trong khi train loss thì tiếp tục giảm, khoảng cách giữa train-val ngày càng lớn, cho thấy dấu hiệu overfit.
- Patch7 có train và val di chuyển đồng thời với nhau, độ lớn train-val không lớn, cả hai còn đi xuống chậm, đây là dấu hiệu của mô hình bị underfit hoặc hạn chế về capacity, chưa bị overfit -> nên còn khả năng để tiếp tục optimize và tuning để tăng khả năng học của mô hình.

In [24]:
# Kiểm định McNemar

from scipy.stats import binomtest

print(f"{'tokenizer':<10}{'agree':>8}{'b':>6}{'c':>6}{'p (McNemar)':>13}")
for tok in TOKENIZERS:
    y, p_s, _ = outputs[f"{tok}_scratch"]
    _, p_t, _ = outputs[f"{tok}_torch"]
    b = int(np.sum((p_s == y) & (p_t != y)))   # scratch đúng, torch sai
    c = int(np.sum((p_s != y) & (p_t == y)))   # scratch sai, torch đúng
    pval = binomtest(b, b + c, 0.5).pvalue if b + c else 1.0
    print(f"{tok:<10}{(p_s == p_t).mean():>8.2%}{b:>6}{c:>6}{pval:>13.3f}")

tokenizer    agree     b     c  p (McNemar)
patch7      93.91%   257   260        0.930
row         92.86%   290   311        0.415
cnn         93.33%   294   285        0.740


Kiểm định **McNemar** lấy nhãn thật và dự đoán của từng model, bỏ qua probs vì kiểm định này chỉ cần biết đúng hay sai.
- `b`: số ảnh scratch đoán đúng nhưng torch sai.
- `c`: ngược lại.
- Những ảnh cả hai cùng đoán đúng, hoặc cùng sai thì không mang thông tin so sánh, nên bỏ qua.

**Giả thuyết H0**: hai model tốt như nhau, khi đó mỗi ảnh "bất đồng" có xác suất 50/50 rơi vào nhóm b hoặc nhóm c, tức là b tuân theo phân phối Binominal(b+c, 0.5)

Qua bảng kết quả, `p` cho xác suất thấy độ lệch giữa b và c lớn nếu hai model thật sự ngang nhau. Nếu p < 0.05 tức là 5% thì mới tạo ra sự khác biệt thống kê, qua đó, giá trị của p ở các tokenzier khác nhau giữa scratch và torch đều lớn hơn nhiều so với 0.05
-> Không có bằng chừng nào cho thấy MSA scratch và torch dùng `nn.MultiheadAttention` cho chất lượng khác nhau.

In [25]:
# Bảng tổng hợp và biểu đồ acc
print(f"{'model':<16}{'N':>4}{'params':>9}{'ep':>4}{'s/ep':>7}{'lat(ms)':>9}"
      f"{'train':>8}{'val':>8}{'test':>8}{'gap':>6}")
for r in results:
    print(f"{r['model']:<16}{r['num_tokens']:>4}{r['params']:>9,}{r['best_epoch']:>4}{r['sec_per_epoch']:>7.1f}"
          f"{r['latency_ms']:>9.2f}{r['train_acc']:>8.2%}{r['val_acc']:>8.2%}{r['test_acc']:>8.2%}"
          f"{(r['train_acc'] - r['val_acc']) * 100:>6.1f}")

e1 = {r["model"]: r for r in json.loads((Path.cwd().parent / "E1/results/results.json").read_text())}

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for r in results:
    color = TOK_COLORS.get(r["tokenizer"], "#8a8984")
    is_scratch = r["attention"] == "scratch"
    for ax, xkey in zip(axes, ["sec_per_epoch", "latency_ms"]):
        ax.scatter(r[xkey], r["test_acc"], color=color, marker="o" if is_scratch else "s", s=r["params"] / 1000)
        ax.annotate(r["model"], (r[xkey], r["test_acc"]), fontsize=6.5,
                    xytext=(4, 3 if is_scratch else -9), textcoords="offset points")
for ax in axes:
    for m, ls in [("MLP", ":"), ("CNN", "--")]:
        ax.axhline(e1[m]["test_acc"], color="#8a8984", ls=ls, lw=1)
        ax.text(ax.get_xlim()[1], e1[m]["test_acc"], f" E1 {m}", va="center", fontsize=7)
    ax.grid(True, axis="y")
axes[0].set(xlabel="Giây / epoch (train, MPS)", ylabel="Test accuracy", title="Accuracy vs. thời gian train")
axes[1].set(xlabel="Latency batch=1 (ms, CPU)", title="Accuracy vs. latency suy luận")
fig.tight_layout()
save_fig(fig, FIG_DIR, "E2_acc_vs_cost")

model              N   params  ep   s/ep  lat(ms)   train     val    test   gap
patch7_scratch    16   72,074  25    2.3     0.29  91.60%  89.52%  88.01%   2.1
patch7_torch      16   72,074  25    2.3     0.27  91.39%  89.20%  88.04%   2.2
row_scratch       28   71,498  36    3.1     0.29  94.12%  90.64%  89.26%   3.5
row_torch         28   71,498  24    2.8     0.26  92.81%  90.57%  89.47%   2.2
cnn_scratch       49   93,962  15    6.4     0.43  93.97%  91.93%  91.13%   2.0
cnn_torch         49   93,962  21    6.3     0.41  94.78%  92.11%  91.04%   2.7


PosixPath('figures/E2_acc_vs_cost.png')

![alt text](figures/E2_acc_vs_cost.png)

Thời gian train mỗi epoch:

- patch7 mất 2.3 s, row 2.8–3.1 s, cnn 6.3–6.4 s.
- Chi phí tăng nhanh hơn số token. Từ N = 16 lên N = 49 (gấp khoảng 3 lần), thời gian tăng khoảng 2.8 lần. Lý do là ma trận attention có kích thước N × N, cộng thêm chi phí của các lớp conv trong stem.
- Ở cùng một tokenizer, scratch và torch gần như bằng nhau, chỉ lệch khoảng 0.1–0.3 s.

Về latency, ở cả ba tokenizer thì bản torch luôn có độ trễ thấp hơn scratch, tức là torch nhanh hơn khoảng 5-10% so với scratch.

So với baseline E1 (MLP và CNN):

- Không model nào vượt qua được CNN của E1
- CNN-stem của E2 là gần với CNN của E1 nhất, nhưng dùng khoảng 94k tham số thay vì dùng khoảng 422k tham số của CNN E1.
- Row với khoảng 71~72k tham số gần bằng với MLP của E1 với hơn 530k tham số.

In [26]:
# Confusion matrix, recall theo lớp và các lỗi tự tin nhất
from common.plotting import plot_misclassified
from E1.report import KEY_PAIRS

scratch_outs = {t: outputs[f"{t}_scratch"] for t in TOKENIZERS}
fig, cms = plot_confusion_matrices(scratch_outs, FASHION_MNIST_CLASSES)
save_fig(fig, FIG_DIR, "E2_confusion_matrices")

print(f"{'Recall (%)':<26}" + "".join(f"{t:>8}" for t in cms))
for k, cls_name in enumerate(FASHION_MNIST_CLASSES):
    print(f"{cls_name:<26}" + "".join(f"{cms[t][k, k]:>8.1%}" for t in cms))
for true_name, pred_name in KEY_PAIRS:
    i, j = FASHION_MNIST_CLASSES.index(true_name), FASHION_MNIST_CLASSES.index(pred_name)
    print(f"{true_name + ' -> ' + pred_name:<26}" + "".join(f"{cms[t][i, j]:>8.0%}" for t in cms))

for t, (yt, yp, pr) in scratch_outs.items():
    save_fig(plot_misclassified(X_test, yt, yp, pr, FASHION_MNIST_CLASSES, f"{t}_scratch"),
             FIG_DIR, f"E2_misclassified_{t}")

Recall (%)                  patch7     row     cnn
T-shirt/top                  85.7%   84.9%   86.0%
Trouser                      97.2%   97.4%   97.3%
Pullover                     81.7%   81.8%   86.0%
Dress                        88.9%   90.8%   91.2%
Coat                         79.5%   86.4%   91.0%
Sandal                       94.3%   97.5%   98.0%
Shirt                        65.7%   66.0%   71.0%
Sneaker                      95.9%   92.9%   97.4%
Bag                          96.1%   97.9%   97.7%
Ankle boot                   95.1%   97.0%   95.7%
Coat -> Pullover               10%      5%      3%
Shirt -> T-shirt/top           15%     15%     10%
T-shirt/top -> Shirt            8%     10%     10%
Shirt -> Pullover               9%      8%      6%
Shirt -> Coat                   6%      7%     10%


![alt text](figures/E2_confusion_matrices.png)

Qua ba confusion matrix ở cả ba tokenizer trên, ta có thể thấy:
- cả ba đều đoán đúng tốt ở các lớp (trouser, sandal, sneaker, bag, boot), đều ≥94%. Tạm gọi đây là các nhóm/lớp dễ, vì các lớp này có hình dạng tổng thể khác biệt rõ.
- với các lớp khác như t-shirt, pullover, coat thì ở mức 80%-90%, đặc biệt là lớp shirt (66%, 66%, 71%). Tạm gọi là các lớp khó, tất cả đều thuộc nhóm áo.
- trong lớp khó, ở các nhóm áo thì CNN cải thiện tốt nhất ở coat (80 -> 86 -> 91%), pullover (82 -> 82 -> 86%) và shirt (66 -> 66 -> 71).
- đối với các cặp dự đoán sai:
  - Đúng là coat (dự đoán thành pullover), thì patch7 tệ nhất, đoán nhầm lên đến 10%, trong khi row và cnn là 5% và 3%.
  - shirt -> coat đây là điểm đáng chú ý, vì CNN có tỉ lệ đoán nhầm cao hơn cả patch7 và row, lần lượt là 6% -> 7% -> 10%.
  - trong đó, cập shirt <-> t-shirt vẫn là cặp có tỉ lệ đoán nhầm cao nhất, tức vẫn là ranh giới khó nhất, giống với kết qua trước đây.

![alt text](figures/E2_misclassified_patch7.png)
![alt text](figures/E2_misclassified_row.png)
![alt text](figures/E2_misclassified_cnn.png)

Lỗi tự tin nhất, có nhiều ảnh mà chính mắt người cũng thấy nhãn đáng ngờ:
- “T: Sneaker” nhưng ảnh trông giống ankle boot cổ cao.
- “T: Shirt” nhưng hình chữ nhật trông giống túi.
- “T: T-shirt” nhưng là áo hai dây, bị đoán thành Dress.

=> Vì các lớp áo có cùng hình dạng tổng thể, muốn phân biệt chúng cần chi tiết cục bộ hơn như cổ áo, khuy, độ dài tay áo... Đây là thứ mà cnn-stem làm tốt, trích xuất tốt hơn một phép chiếu tuyến tính (patch, row). Điều này giải thích vì sao lợi thế cnn tập trung ở nhóm áo.

In [27]:
# Attention maps
def build_model(tokenizer_name, attention="scratch", **overrides):
    cfg = {**E2_CONFIG, **overrides}
    return TransformerClassifier(tokenizer=TOKENIZERS[tokenizer_name](cfg["embed_dim"]),
                                 attention=attention, **cfg)

def get_model(tokenizer_name, attention="scratch"):
    name = f"{tokenizer_name}_{attention}"
    ckpt = RESULTS_DIR / f"model_{name}.pt"
    model = build_model(tokenizer_name, attention)
    if ckpt.exists():
        model.load_state_dict(torch.load(ckpt, map_location="cpu"))
    else:
        print(f"[{name}] chưa có checkpoint -> train lại (số liệu có thể lệch nhẹ so với results.json)")
        _, _, _, model, _ = run_experiment(tokenizer_name, attention)
        torch.save(model.state_dict(), ckpt)
    return model.to(DEVICE).eval()

TOKEN_GRID = {"patch7": (4, 4), "row": (28, 1), "cnn": (7, 7), "patch4": (7, 7), "overlap": (7, 7)}

@torch.no_grad()
def cls_attention_rollout(model, x):
    _, attn_maps = model(x.to(DEVICE), return_attn=True)
    T = attn_maps[0].shape[-1]
    eye = torch.eye(T, device=DEVICE)
    rollout = eye.expand(x.size(0), T, T)
    for A in attn_maps:
        A = A.mean(dim=1)                       # trung bình các head: [B, T, T]
        A = 0.5 * A + 0.5 * eye                 # tính cả residual
        A = A / A.sum(dim=-1, keepdim=True)
        rollout = A @ rollout
    return rollout[:, 0, 1:]                    # CLS -> các token ảnh: [B, N]

def plot_attention(model_by_tok, idx):
    x = datasets["test"].tensors[0][idx]
    fig, axes = plt.subplots(len(model_by_tok) + 1, len(idx),
                             figsize=(len(idx) * 1.3, (len(model_by_tok) + 1) * 1.4))
    for j, i in enumerate(idx):
        axes[0, j].imshow(X_test[i], cmap="gray")
        axes[0, j].set_title(FASHION_MNIST_CLASSES[y_test[i]], fontsize=7)
    for r, (tok, model) in enumerate(model_by_tok.items(), start=1):
        att = cls_attention_rollout(model, x).reshape(len(idx), 1, *TOKEN_GRID[tok])
        att = F.interpolate(att, size=(28, 28), mode="nearest").squeeze(1).cpu().numpy()
        for j, i in enumerate(idx):
            axes[r, j].imshow(X_test[i], cmap="gray")
            axes[r, j].imshow(att[j], cmap="jet", alpha=0.45)
        axes[r, 0].set_ylabel(tok, fontsize=8)
    for ax in axes.flat:
        ax.set_xticks([]); ax.set_yticks([])
    fig.tight_layout()
    return fig

models_attn = {t: get_model(t) for t in TOKENIZERS}       # ~25 phút nếu phải train lại cả 3
pick = [int(np.flatnonzero(y_test == k)[0]) for k in range(10)]
save_fig(plot_attention(models_attn, pick), FIG_DIR, "E2_attention_rollout")

PosixPath('figures/E2_attention_rollout.png')

![alt text](figures/E2_attention_rollout.png)

Mỗi cột là một ảnh (một lớp ảnh), mỗi hàng là một tokenzier. Màu nóng cho viết token đó đóng góp nhiều vào biểu diễn của cls.

- Với row:

  - T-Shirt: dãy sáng tập trung vào hàng tay áo kết thúc, cho thấy độ dài tay áo là điểm phân biệt của T-shirt với Pullover.
  - Coat: dãy sáng tập trung ở phần cổ áo.
  - Trouser: dãy sáng tập trung ở phần thân quần, nửa dưới, thể hiện độ dài của quần.
  - Sandal, Sneaker, Boot: dãy sáng tập trung vào hàng đế và cổ giày.

- Với patch:

  - Attention khá phân tán, thường rơi vào các pattern nằm ở viền của vật thể.
  - Điều này cũng phản ánh độ thô và thưa của lưới 4x4 với mỗi ô patch là 7x7, không đủ chi tiết để khoanh vùng cho chi tiết.

- Với cnn:

  - Chia thành nhiều ô lưới hơn patch, nhưng attention cũng khá thưa, nhưng có những điểm tập trung nhất định vào chi tiết của các vật thể.

